# 第 10 章の講義と実習　置換検定

講義では，飲用後の二群に見えた差が，無作為な割付だけでも出るかを調べます。後半の約 20 分は，一人ずつの飲用前後の変化に問いを替え，何が同じで何が変わるかを比べます。

## 準備

43 人の実験では，ビール群と水群の飲用後の捕獲数に差が見えました。その差は飲み物の効果を示すのでしょうか。まず講義で，飲み物が各人の飲用後の捕獲数を変えないと仮定した場合に，無作為な割付だけで同じくらいの差が出るかを調べます。実習では問いを「飲用前から飲用後までの変化は，二群でどれほど違うか」に替えます。

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。

CSV の 1 行は一人について前か後に行った一回の測定です。準備セルは同じ `volunteer` の行を対応させ，一人一行の `paired` を作ります。捕獲数は，その測定で放した 50 匹のうち二つのトラップに入った数の合計です。くじのモデルでは，43 人からビール群の 25 人を選ぶどの組も等確率とします。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

file_name = "mosquito-experiment.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

experiment = pd.read_csv(csv_path)
before = experiment.loc[experiment["test"] == "before", ["volunteer", "activated"]]
after = experiment.loc[experiment["test"] == "after", ["volunteer", "group", "activated"]]
paired = after.merge(
    before, on="volunteer", suffixes=("_after", "_before"), validate="one_to_one"
)
drinks = paired["group"].to_numpy()
n_beer = (drinks == "beer").sum()
n_water = (drinks == "water").sum()
print(f"Participants: {len(paired)} (beer {n_beer}, water {n_water})")
paired[["volunteer", "group", "activated_before", "activated_after"]].head(3)

## 講義中に実行するセル

見出しの順に実行します。セルを動かす前に，比べる行・値・図を確認します。

### 1. 実際に割り当てた二群の平均を比べる

最初に，実際に割り当てられたビール群と水群の飲用後の捕獲数を比べます。群の人数が違うため，一人当たりの平均をそれぞれ求め，「ビール群 − 水群」で観測差を計算します。下の図の一点は一人の飲用後の測定です。平均の差だけでなく，個人の値が両群でどの範囲にあるかも見てください。この観測差が，割付だけでどれほど起こり得るかを次に考えます。

In [ ]:
lecture_groups = paired.groupby("group")["activated_after"]
lecture_summary = lecture_groups.agg(participants="size", total="sum", mean="mean")
lecture_observed_gap = lecture_summary.loc["beer", "mean"] - lecture_summary.loc["water", "mean"]
print(f"Beer − water: {lecture_observed_gap:+.2f} mosquitoes per participant")
lecture_summary.round(2)

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, sharey=True, figsize=(6.4, 3.6),
                         layout="constrained")
for ax, drink, color, marker in zip(
    axes, ["beer", "water"], ["#286b70", "#ad522f"], ["o", "s"],
):
    lecture_group_counts = lecture_groups.get_group(drink)
    lecture_stack = lecture_group_counts.groupby(lecture_group_counts).cumcount() + 1
    ax.scatter(lecture_group_counts, lecture_stack, color=color, marker=marker, s=29)
    ax.axvline(lecture_group_counts.mean(), color="#3c4854", linestyle="--")
    ax.set(title=f"{drink.title()} (n = {len(lecture_group_counts)})", yticks=[1, 2, 3, 4])
axes[-1].set(xlabel="Mosquitoes caught per participant", xlim=(0, 50))
fig.supylabel("Stacked participants")
plt.show()

### 2. 4 人で引けるくじを全部書き出す

43 人の札を配り直す前に，4 人の架空の例で計算の理由を確かめます。A、B、C、D の捕獲数は 12、18、21、29 匹とします。ここでは**各人の捕獲数は，どちらの飲み物を割り当てても同じ**と仮定します。だから，捕獲数は固定したままビールの札を引く二人だけを替えられます。C と D がビールの札を引いたときの差は +10 匹です。すべての割付を並べ，0 から同じ距離かそれ以上離れる差が何通りあるか数えてください。

In [ ]:
lecture_toy_counts = pd.Series([12, 18, 21, 29], index=list("ABCD"))
lecture_toy_gaps = {}
for beer_pair in ["AB", "AC", "AD", "BC", "BD", "CD"]:
    has_beer = lecture_toy_counts.index.isin(list(beer_pair))
    lecture_toy_gaps[beer_pair] = (
        lecture_toy_counts[has_beer].mean() - lecture_toy_counts[~has_beer].mean()
    )
pd.Series(lecture_toy_gaps, name="Beer − water (mosquitoes)")

### 3. 43 人の札を一回だけ引き直す

4 人の例と同じ仮定を，実際の 43 人に置きます。各人の飲用後の捕獲数が割付で変わらないなら，その数を固定して札だけ配り直した場合の差を計算できます。これは「ビールの効果が一人では正，別の人では負で，平均だけ 0」という条件より強い仮定です。元の `group` と新しい `shuffled_group` を比べ，札が変わった人と各群の人数を確かめます。

In [ ]:
lecture_after_counts = paired["activated_after"].to_numpy()
lecture_rng = np.random.default_rng(2026)
lecture_shuffled = lecture_rng.permutation(drinks)

lecture_first_draw = paired[["volunteer", "activated_after", "group"]].copy()
lecture_first_draw["shuffled_group"] = lecture_shuffled
print("Beer:", (lecture_shuffled == "beer").sum(),
      "Water:", (lecture_shuffled == "water").sum())
lecture_first_draw.head(5)

### 4. 10,000 回の割付から差を一つずつ集める

各人の捕獲数を固定し，25 人にビール，残る 18 人に水の札を配る操作を 10,000 回繰り返します。一回の割付から二群の平均差を一つ保存し，「各人に飲み物の効果がない」という仮定のもとで出る差の分布を作ります。先頭の差は直前のセルで見た札の並びから求めたものです。シードを一度指定し，続きの乱数で配り直します。

In [ ]:
lecture_rng = np.random.default_rng(2026)
lecture_repetitions = 10_000
lecture_draw_gaps = []
for _ in range(lecture_repetitions):
    lecture_shuffled = lecture_rng.permutation(drinks)
    lecture_beer = lecture_after_counts[lecture_shuffled == "beer"]
    lecture_water = lecture_after_counts[lecture_shuffled == "water"]
    lecture_draw_gaps.append(lecture_beer.mean() - lecture_water.mean())
lecture_null_gaps = np.array(lecture_draw_gaps)
print("Draws:", len(lecture_null_gaps), "First three:", np.round(lecture_null_gaps[:3], 2))

### 5. 観測差以上に 0 から離れた回を数える

帰無仮説のもとで札を配り直した差が，実際の観測差ほど珍しいかを調べます。破線は観測差と同じだけ 0 から離れた左右二か所です。差の符号がどちらでも，絶対値が観測差以上なら数に入れます。左と右の回数を合わせ，観測した割付の 1 回を分子と分母に加えて割合を求めます。この割合は，帰無仮説のもとで観測差以上に極端な差が出る頻度を表します。

In [ ]:
lecture_boundary = abs(lecture_observed_gap)
lecture_left_end = lecture_null_gaps <= -lecture_boundary
lecture_right_end = lecture_null_gaps >= lecture_boundary
lecture_tail_count = lecture_left_end.sum() + lecture_right_end.sum()
lecture_p_value = (lecture_tail_count + 1) / (lecture_repetitions + 1)
print("Left:", lecture_left_end.sum(), "Right:", lecture_right_end.sum())
print(f"Two-sided p: {lecture_p_value:.4f}")

In [ ]:
lecture_in_tail = lecture_left_end | lecture_right_end
fig, ax = plt.subplots(layout="constrained")
ax.hist(
    [lecture_null_gaps[~lecture_in_tail], lecture_null_gaps[lecture_in_tail]],
    bins=np.arange(-12.5, 13.5, 1), stacked=True,
    color=["#8598a5", "#a04b2b"], edgecolor="white",
)
ax.axvline(-lecture_boundary, color="#3c4854", linestyle="--")
ax.axvline(lecture_boundary, color="#3c4854", linestyle="--")
ax.set(xlabel="Difference in group means (mosquitoes)", ylabel="Draws", xlim=(-12.5, 12.5))
plt.show()

### 6. 観測差を匹数と割合で伝える

観測差が割付だけでも起こり得る頻度に加え，何匹分の差かも報告します。各人の測定で放した蚊は 50 匹です。平均捕獲数の差を 50 で割って百分率に直せば，捕獲割合の差をパーセントポイントで表せます。くじで得た差も同じ単位に直したとき，観測差以上に極端な回数が変わるか考えてください。

In [ ]:
lecture_gap_in_points = 100 * lecture_observed_gap / 50
print(f"Observed difference: {lecture_observed_gap:+.2f} mosquitoes")
print(f"Observed difference: {lecture_gap_in_points:+.2f} percentage points")

## 自分で分析する実習（約 20 分）

### 1. 前後を対応させて考える（約 5 分）

講義では**飲用後の捕獲数**を二群で比べました。ここからは各人の**飲用前から飲用後への変化**を比べます。上の表で，ビール群の `subj1` は 16 匹から 14 匹に，`subj3` は 15 匹から 27 匹になりました。二人の「後 − 前」を手で計算し，同じ飲み物でも変化の向きが違うことを一文で書きます。水群にも自然な前後変化があり得るため，水群についても同じ引き算をします。

次のセルでは，準備済みの `def` から始まる関数を使い，講義の**飲用後の数**の比較を再現します。関数は札を配り直すたびにビール群と水群の平均差を記録し，`return` で差の配列を返します。まず講義と同じ観測差と両側の割合が出ることを確認してください。この結果が，後で前後の変化を使った結果との比較基準になります。

前後の変化に替えると，飲用前の二群の平均差が，飲用後の二群の平均差から差し引かれます。ただし，ビール群と水群の変化の向きや大きさは，結果を見るまで分かりません。飲用後の数だけを比べた場合と比べ，観測差がどう変わりそうかを予想してください。くじは人ごとに配られました。前後の測定行を別々に動かす案と，一人の前後差を保って人ごとに札を動かす案のどちらが元の割付を再現するかも，理由を一文で書いてください。

（ここに二人の変化，観測差の予想，札を動かす単位と理由を記入）

In [ ]:
def simulate_differences(counts, drinks, n_draws, seed):
    rng = np.random.default_rng(seed)
    differences = []
    for _ in range(n_draws):
        shuffled_drinks = rng.permutation(drinks)
        beer = counts[shuffled_drinks == "beer"]
        water = counts[shuffled_drinks == "water"]
        differences.append(beer.mean() - water.mean())
    return np.array(differences)


n_draws = 10_000
after_counts = paired["activated_after"].to_numpy()
beer_after = after_counts[drinks == "beer"]
water_after = after_counts[drinks == "water"]
observed_after = beer_after.mean() - water_after.mean()
null_after = simulate_differences(after_counts, drinks, n_draws, seed=2026)
after_tail = np.abs(null_after) >= abs(observed_after)
p_after = (after_tail.sum() + 1) / (n_draws + 1)

print(f"After: {observed_after:+.2f} mosquitoes")
print(f"Two-sided p: {p_after:.4f} ({after_tail.sum()} of {n_draws} draws)")

### 2. 変化量の差を評価する（約 9 分）

一人ずつの「後 − 前」を計算し，`change_counts` に配列として入れます。最初のセルに一行書き，先頭の 3 人の値を手計算と照合してください。その下のセルでは，実際に割り当てられた二群の変化の平均を比べ，同じ 43 人に札を配り直した 10,000 回分の差も計算します。ここで固定するのは**各人の前後差**です。各人の飲用後の数を固定した講義の検定とは，比べる値と問いが違います。

`null_change` は札を配り直して得た変化の平均差 10,000 個，`observed_change` は実際の割付での変化の平均差です。帰無仮説のもとで観測差以上に極端な回を数えるため，0 からの距離が観測差と同じかそれ以上なら `True` とする一行を `change_tail` に書いてください。左右両側と境界の値を含めます。両方の行を書き終えると，飲用後の数と前後の変化による二つの分布が同じ軸で表示されます。

In [ ]:
change_counts = None
# 同じ人の「後 − 前」を計算し，.to_numpy() で配列にする。

if change_counts is None:
    print("前後差を求める一行を書いてから，このセルを実行します。")
else:
    print("First three changes:", change_counts[:3])

In [ ]:
change_tail = None
if change_counts is not None:
    beer_change = change_counts[drinks == "beer"]
    water_change = change_counts[drinks == "water"]
    observed_change = beer_change.mean() - water_change.mean()
    null_change = simulate_differences(change_counts, drinks, n_draws, seed=2026)
    print(f"Beer mean change: {beer_change.mean():+.2f}")
    print(f"Water mean change: {water_change.mean():+.2f}")
    print(f"Difference: {observed_change:+.2f}")
    # 距離が観測差と同じかそれ以上の位置を True とする一行を書く。

if change_tail is None:
    print("前後差と両側の判定を書くと，比較図が出ます。")
else:
    p_change = (change_tail.sum() + 1) / (n_draws + 1)
    print(f"Two-sided p: {p_change:.4f} ({change_tail.sum()} of {n_draws} draws)")

    fig, axes = plt.subplots(2, 1, sharex=True, sharey=True, layout="constrained")
    bins = np.arange(-12.5, 13.5, 1)
    for ax, title, simulated, observed, in_tail in [
        (axes[0], "After only", null_after, observed_after, after_tail),
        (axes[1], "After − before", null_change, observed_change, change_tail),
    ]:
        ax.hist(
            [simulated[~in_tail], simulated[in_tail]],
            bins=bins, stacked=True, color=["#8598a5", "#a04b2b"], edgecolor="white",
        )
        ax.axvline(-abs(observed), color="#3c4854", linestyle="--")
        ax.axvline(abs(observed), color="#3c4854", linestyle="--")
        ax.set_title(title, loc="left")
    fig.supylabel("Draws")
    axes[-1].set(
        xlabel="Difference in group means (mosquitoes)",
        xlim=(-12.5, 12.5),
    )
    plt.show()

<details>
<summary>前後差と両側の判定の書き方</summary>

前後差を求めて配列にする一行を，`change_counts = None` の下に書きます。

```python
change_counts = (paired["activated_after"] - paired["activated_before"]).to_numpy()
```

`if change_counts is not None:` の中で，次の一行を書きます。

```python
    change_tail = np.abs(null_change) >= abs(observed_change)
```

</details>

### 3. 二つの問いを読み比べる（約 6 分）

講義と実習では，札を配り直す方法は同じでも，固定して比べた値が違います。二段の図の破線はそれぞれの問いでの観測差と同じ距離，濃い部分はその距離以上に離れた割付の回数です。二つの問いに対する答えがどう違うか，図と出力の数を示して 3〜4 文で報告してください。

1. 前後差は，割付ごとに平均で何匹だったか。ビール群と水群の差は何匹で，帰無仮説のもとでその差と同じかそれ以上の大きさが出たのは何回か。
2. 飲用後の捕獲数の比較から前後の変化の比較へ，何を変え，何を固定したか。二つの図で破線の位置や濃い部分を読み比べて説明する。
3. 前後差をブートストラップする場合に変わるものと，今回くじを引き直したときに変わるものを，`subj1` の前後差を例に説明する。

（ここに報告を記入）

提出前にカーネルを再起動し，上から全セルを実行します。図と記入した説明を残したノートブックを保存します。

#### 別のくじで 10,000 回（任意）

`null_change` を計算する行の `seed=2026` を `seed=2027` に替えます。元の観測差を揃えたまま，再び左右の端の件数を合わせて比べます。どちらの数字が変わり，どちらが同じだったか説明してください。

#### 出典

データは Thierry Lefèvre が提供した配布 CSV を使います。データと派生物は CC BY です。原著では前後の測定と他の条件も含むモデルを用いました。[データの来歴と利用条件](https://github.com/matthew-brett/datasets/tree/master/mosquito_beer) ・ [Lefèvre ほかの原著](https://doi.org/10.1371/journal.pone.0009546)